# FIAP — Tech Challenge Fase 2
## 03.3 — Gold Estados

Integra:

- indicadores Silver estaduais;
- metas por UF;
- indicadores da Gold Alunos por UF.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Leitura e integração

In [0]:
# ============================================================
# LEITURA E INTEGRAÇÃO DAS BASES GOLD POR ESTADO
# ============================================================

metadata = pd.read_parquet(
    CONFIG_PATH / "gold_metadata"
)

df_alunos_uf = ler_csv(
    GOLD_PATH
    / "alunos_ufs"
    / "GOLD_ALUNOS_UFS.csv"
)

bases = []

for ano in [2023, 2024, 2025]:

    print("=" * 100)
    print(f"Processando ano: {ano}")

    # --------------------------------------------------------
    # Metadados dos arquivos Silver
    # --------------------------------------------------------

    meta_estados = (
        metadata[
            (metadata["produto"] == "gold_estados")
            & (metadata["dataset"] == "estados")
            & (metadata["ano"] == ano)
        ]
        .iloc[0]
    )

    meta_metas = (
        metadata[
            (metadata["produto"] == "gold_estados")
            & (metadata["dataset"] == "metas_ufs")
            & (metadata["ano"] == ano)
        ]
        .iloc[0]
    )

    # --------------------------------------------------------
    # Leitura dos arquivos
    # --------------------------------------------------------

    df_estados = ler_csv(
        Path(meta_estados["silver_path"])
        / meta_estados["silver_file_name"]
    )

    df_metas = ler_csv(
        Path(meta_metas["silver_path"])
        / meta_metas["silver_file_name"]
    )

    df_alunos = (
        df_alunos_uf[
            df_alunos_uf["ANO"] == ano
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Inclusão/garantia do ano
    # --------------------------------------------------------

    df_estados["ANO"] = ano
    df_metas["ANO"] = ano
    df_alunos["ANO"] = ano

    # --------------------------------------------------------
    # Padronização das colunas da base de metas
    # --------------------------------------------------------

    # A Silver pode ter preservado CD_UF ou já renomeado para CO_UF.
    if "CD_UF" in df_metas.columns and "CO_UF" not in df_metas.columns:
        df_metas = df_metas.rename(
            columns={"CD_UF": "CO_UF"}
        )

    # A Silver pode ter preservado SIGLA_UF.
    if "SIGLA_UF" in df_metas.columns and "SG_UF" not in df_metas.columns:
        df_metas = df_metas.rename(
            columns={"SIGLA_UF": "SG_UF_META"}
        )

    # --------------------------------------------------------
    # Validação das chaves obrigatórias
    # --------------------------------------------------------

    chaves_estados = {"ANO", "CO_UF"}
    chaves_metas = {"ANO", "CO_UF"}
    chaves_alunos = {"ANO", "CO_UF"}

    faltantes_estados = (
        chaves_estados
        - set(df_estados.columns)
    )

    faltantes_metas = (
        chaves_metas
        - set(df_metas.columns)
    )

    faltantes_alunos = (
        chaves_alunos
        - set(df_alunos.columns)
    )

    if faltantes_estados:
        raise KeyError(
            f"Colunas ausentes em estados {ano}: "
            f"{sorted(faltantes_estados)}"
        )

    if faltantes_metas:
        raise KeyError(
            f"Colunas ausentes em metas_ufs {ano}: "
            f"{sorted(faltantes_metas)}"
        )

    if faltantes_alunos:
        raise KeyError(
            f"Colunas ausentes em gold_alunos_uf {ano}: "
            f"{sorted(faltantes_alunos)}"
        )

    # --------------------------------------------------------
    # Normalização das chaves
    # --------------------------------------------------------

    df_estados["CO_UF"] = normalizar_codigo(
        df_estados["CO_UF"]
    )

    df_metas["CO_UF"] = normalizar_codigo(
        df_metas["CO_UF"]
    )

    df_alunos["CO_UF"] = normalizar_codigo(
        df_alunos["CO_UF"]
    )

    # --------------------------------------------------------
    # Remoção do registro agregado Brasil
    # --------------------------------------------------------

    df_metas = (
        df_metas[
            df_metas["CO_UF"].notna()
            & (
                df_metas["CO_UF"]
                .astype(str)
                .str.lower()
                != "nan"
            )
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Integração Estados + Metas
    # --------------------------------------------------------

    df_integrado = df_estados.merge(
        df_metas,
        on=["ANO", "CO_UF"],
        how="left",
        suffixes=("", "_META"),
        indicator="cobertura_metas"
    )

    df_integrado["cobertura_metas"] = (
        df_integrado["cobertura_metas"]
        .map({
            "both": "COM_META",
            "left_only": "SEM_META",
            "right_only": "SOMENTE_META"
        })
        .astype(str)
    )

    # --------------------------------------------------------
    # Integração com indicadores Gold de alunos por UF
    # --------------------------------------------------------

    colunas_alunos_remover = [
        coluna
        for coluna in ["SG_UF"]
        if coluna in df_alunos.columns
    ]

    df_integrado = df_integrado.merge(
        df_alunos.drop(
            columns=colunas_alunos_remover,
            errors="ignore"
        ),
        on=["ANO", "CO_UF"],
        how="left",
        suffixes=("", "_ALUNOS"),
        indicator="cobertura_alunos"
    )

    df_integrado["cobertura_alunos"] = (
        df_integrado["cobertura_alunos"]
        .map({
            "both": "COM_INDICADORES_ALUNOS",
            "left_only": "SEM_INDICADORES_ALUNOS",
            "right_only": "SOMENTE_ALUNOS"
        })
        .astype(str)
    )

    # --------------------------------------------------------
    # Validação da integração
    # --------------------------------------------------------

    print("Colunas estados:", list(df_estados.columns))
    print("Colunas metas:", list(df_metas.columns))
    print("Colunas alunos:", list(df_alunos.columns))

    print(
        "Registros integrados:",
        len(df_integrado)
    )

    print(
        "Estados sem meta:",
        (
            df_integrado["cobertura_metas"]
            == "SEM_META"
        ).sum()
    )

    print(
        "Estados sem indicadores de alunos:",
        (
            df_integrado["cobertura_alunos"]
            == "SEM_INDICADORES_ALUNOS"
        ).sum()
    )

    bases.append(df_integrado)

# ============================================================
# CONSOLIDAÇÃO DOS ANOS
# ============================================================

df_gold_estados = pd.concat(
    bases,
    ignore_index=True
)

if df_gold_estados.empty:
    print(
        "Nenhum registro foi gerado "
        "para a Gold Estados."
    )
else:
    display(
        df_gold_estados.head()
    )

## 5. Indicadores

In [0]:
for coluna in ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP", "META_FINAL_2030"]:
    if coluna in df_gold_estados.columns:
        df_gold_estados[coluna] = converter_numero(df_gold_estados[coluna])

df_gold_estados["gap_meta_2030"] = (
    df_gold_estados["META_FINAL_2030"]
    - df_gold_estados["PC_ALUNO_ALFABETIZADO"]
)

df_gold_estados["risco_educacional"] = pd.cut(
    df_gold_estados["PC_ALUNO_ALFABETIZADO"],
    bins=[-np.inf, 50, 70, 85, np.inf],
    labels=["Crítico", "Alto", "Médio", "Baixo"]
)

df_gold_estados["_gold_processed_at"] = datetime.now().isoformat()

## 6. Persistência

In [0]:
for ano in [2023, 2024, 2025]:
    registro = metadata[
        (metadata["produto"] == "gold_estados")
        & (metadata["dataset"] == "estados")
        & (metadata["ano"] == ano)
    ].iloc[0]

    salvar_csv(
        df_gold_estados[df_gold_estados["ANO"] == ano],
        Path(registro["gold_output_path"]),
        registro["gold_file_name"]
    )

print("Gold Estados salva com sucesso.")